# Week 33, Day 6: Forge: Week 33 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w033/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
print("1.", round(1 * 10 / 9 * np.log(10), 2), "ms")
print("2.", 1000 / (0.4 * 500), "Hz")
print("3.", round(1000 / np.sqrt(0.1 * 100 * 400), 1), "Hz")

### Exercise: Part A: fit and classify

Write `fit(amps)`: search U from 0.02 to 0.8 (step 0.02), τ_D in (50, 100, 200, 400, 800, 1200) ms and τ_F in (0, 50, 100, 200, 400, 800, 1600) ms; for each, compute the model responses m, the scale A = (m · amps)/(m · m) and the squared error; return the best (error, U, τ_D, τ_F, A). Then classify each cell: facilitating if the fitted second response exceeds the first, otherwise depressing.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def tm(times, U, tauD, tauF):
    u, x, last, out = U, 1.0, None, []
    for t in times:
        if last is not None:
            d = t - last
            x = 1 + (x - 1) * np.exp(-d / tauD)
            u = (U + (u - U) * np.exp(-d / tauF)) if tauF > 0 else U
        out.append(u * x)
        x = x - u * x
        if tauF > 0:
            u = u + U * (1 - u)
        last = t
    return np.array(out)

r = np.random.default_rng(97)
train = np.cumsum(r.exponential(1000 / 12, 50))
hidden = {"cell 1": (0.06, 120.0, 900.0, 3.0), "cell 2": (0.55, 700.0, 0.0, 1.5)}
resp = {c: A * tm(train, U, tD, tF) * (1 + 0.08 * r.normal(size=50)) for c, (U, tD, tF, A) in hidden.items()}

def fit(amps):
    best = (np.inf, None, None, None, None)
    for U in np.arange(0.02, 0.81, 0.02):
        for tD in (50, 100, 200, 400, 800, 1200):
            for tF in (0, 50, 100, 200, 400, 800, 1600):
                ...
    return best

fits, kind = {}, {}
for c in resp:
    fits[c] = fit(resp[c])
    m = tm(train, *fits[c][1:4])
    kind[c] = ...
    print(f"{c}: U = {fits[c][1]:.2f}, tau_D = {fits[c][2]}, tau_F = {fits[c][3]}, A = {fits[c][4]:.2f}: {kind[c]}")

In [ ]:
import numpy as np
def test_fit():
    "fit returns (error, U, tauD, tauF, A) with A solved by least squares"
    e, U, tD, tF, A = fits["cell 1"]
    m = tm(train, U, tD, tF)
    assert abs(A - (m @ resp["cell 1"]) / (m @ m)) < 1e-9

def test_classes():
    "one connection is classified as depressing and the other as facilitating, with sensible parameters"
    assert sorted(kind.values()) == ["depressing", "facilitating"]
    dep = [k for k in kind if kind[k] == "depressing"][0]; fac = [k for k in kind if kind[k] == "facilitating"][0]
    assert fits[dep][1] > 0.3 and fits[fac][1] < 0.2 and fits[fac][3] >= 200

test_fit()
test_classes()
print("All checks passed")

### Exercise: Part B: held-out prediction

Build the test train: 10 spikes at 3 Hz, 10 at 60 Hz, then 20 at 25 Hz. Record the hidden connections' responses to it (with 8% noise, using the same rng), predict them with the fitted parameters, and compute the R² of each prediction in `r2[c]`.

In [ ]:
test = np.cumsum(np.r_[np.full(10, 1000 / 3), np.full(10, 1000 / 60), np.full(20, 1000 / 25)])
new_resp, r2 = {}, {}
for c, (U, tD, tF, A) in hidden.items():
    new_resp[c] = A * tm(test, U, tD, tF) * (1 + 0.08 * r.normal(size=test.size))
    pred = ...
    r2[c] = ...
    print(f"{c} ({kind[c]}): held-out R^2 = {r2[c]:.2f}")
    plt.plot(new_resp[c], "o", label=f"{c} recorded"); plt.plot(pred, label=f"{c} predicted")
plt.axvline(9.5, color="gray", lw=0.5); plt.axvline(19.5, color="gray", lw=0.5)
plt.xlabel("spike number"); plt.ylabel("response"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_r2():
    "both fitted models explain over 70% of the variance of the held-out responses"
    assert all(r2[c] > 0.7 for c in r2)

def test_message():
    "the depressing connection responds most at the start of the burst, the facilitating one later in the train"
    dep = [k for k in kind if kind[k] == "depressing"][0]; fac = [k for k in kind if kind[k] == "facilitating"][0]
    assert np.argmax(new_resp[dep]) <= 11 and np.argmax(new_resp[fac]) > 11

test_r2()
test_message()
print("All checks passed")